# 03 - Analisis Sentimen Tingkat Ulasan

**Tahap 5 (bagian pertama).** Setiap ulasan diberi polaritas berdasarkan
**isi teksnya**, lalu hasilnya **dibandingkan dengan rating** untuk melihat
seberapa jauh teks mencerminkan rating.

## Mengapa label diambil dari teks, bukan rating?

Rating tidak selalu menggambarkan isi ulasan. Pada dataset ini ditemukan:

| Rating | Isi ulasan | Keluhan yang tersembunyi |
|---|---|---|
| 4 bintang | "Live musik tiap hari. Bumbu kurang meresap" | cita rasa |
| 5 bintang | "Menu rawon, asin. Mohon diperhatikan" | cita rasa |
| 5 bintang | "Suasana ok, makanan tidak terlalu ok, pelayanan kurang memuaskan" | makanan & pelayanan |
| 5 bintang | "Pesen nachos lebih dari 45 menit baru disajikan" | kecepatan |
| 1 bintang | "Pelayanan cukup baik... soal makanan ndak worthed" | pujian pelayanan hilang |

Bila sentimen diambil dari rating, seluruh keluhan pada ulasan 4-5 bintang
otomatis hilang. Karena itu:

- **Label sentimen = hasil analisis teks** (metode lexicon InSet).
- **Rating = acuan pembanding** untuk mengukur kesesuaian dan menemukan
  ulasan yang isinya bertentangan dengan ratingnya (*hidden complaint*).

In [1]:
import sys
from pathlib import Path

# Menemukan root proyek agar `import src` berjalan dari mana pun notebook dibuka.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import config

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 90)

print("Root proyek:", ROOT)

Root proyek: D:\hikam\tugas DataMining


## 3.1 Memuat data hasil preprocessing

In [2]:
from src import io_utils, lexicon, sentiment

df = io_utils.load_csv(config.PREP_CSV)
print(f"Ulasan: {len(df):,}")
df[["review_id", "Rating", "text_normalized", "tokens_stemmed"]].head(3)

Ulasan: 6,795


,review_id,Rating,text_normalized,tokens_stemmed
0,1,5,ramai banget kalau setiap datang ke sini tapi tetap dicariin kursi kok sama pelayan ma...,ramai dicariin kursi layan makan minum enak harga golong affordable live musik asik no...
1,2,1,mbah parkiranya nyebelin banget sudah bayar kok ditagih lagi mana tidak bantuin narik ...,mbah parkiranya nyebelin bayar tagih bantuin narik motor susah
2,3,3,pengalaman sudah laper pengen makan banget pesan nasgor magelangan tapi rasa nasgornya...,alam laper makan pesan nasgor magelangan nasgornya astaghfirullah nasi nasi murah basi...


## 3.2 Menyiapkan lexicon InSet

Lexicon InSet berisi daftar kata positif dan negatif beserta bobotnya.
Karena banyak kata muncul di **kedua** file (mis. `enak` ada di positif +5 dan
negatif -4), resolusi konflik memakai bobot absolut terbesar. Kata fungsi dan
frasa multi-kata dibuang agar tidak menimbulkan polaritas palsu.

In [3]:
lex = lexicon.build_domain_lexicon()
io_utils.save_csv(lexicon.lexicon_frame(lex), config.TABLES_DIR / "03_lexicon_digunakan.csv")
lexicon.lexicon_summary(lex)

,Metrik,Nilai
0,Jumlah kata,7783.000
1,Kata positif,2626.000
2,Kata negatif,5157.000
3,Skor maksimum,5.000
4,Skor minimum,-5.000
5,Rata-rata skor,-1.368


In [4]:
lexicon.lexicon_frame(lex).head(15)

,Kata,Skor,Polaritas
0,adika,5.0,Positif
1,agam,5.0,Positif
2,alhamdulillah,5.0,Positif
3,allah,5.0,Positif
4,amanah,5.0,Positif
5,anakan,5.0,Positif
6,andal,5.0,Positif
7,apas,5.0,Positif
8,awat,5.0,Positif
9,bagak,5.0,Positif


In [5]:
pd.DataFrame(
    [{"Kata": k, "Skor": lex[k]} for k in
     ["enak", "nyaman", "ramah", "bersih", "murah", "mahal", "kotor",
      "lama", "lambat", "kecewa", "bau", "worth"] if k in lex]
)

,Kata,Skor
0,enak,4
1,nyaman,3
2,ramah,4
3,bersih,3
4,murah,2
5,mahal,-3
6,kotor,-4
7,lama,-3
8,lambat,-3
9,kecewa,-4


## 3.3 Skoring sentimen

Aturan skoring:

- Ulasan dipecah menjadi klausa (`.` `,` `;` `tapi` `namun` `sayangnya` dst.).
- Skor tiap klausa = jumlah bobot kata sentimen.
- **Negasi** (`tidak`, `kurang`, `gak`, `bukan`, ...) membalik polaritas kata
  hingga dua token di belakangnya.
- **Penguat** (`sangat`, `banget`, `sekali`) mengalikan bobot.
- Skor dokumen = total skor seluruh klausa.

Label: skor > 0 → Positif, skor < 0 → Negatif, skor = 0 → Netral.

In [6]:
scored, tables = sentiment.run_document_sentiment(df, lex)
io_utils.save_csv(scored, config.DOC_SENTIMENT_CSV)

tables["sentiment_distribution"]

,Sentimen,Jumlah,Persentase
0,Positif,4610,67.84
1,Netral,1275,18.76
2,Negatif,910,13.39


In [7]:
from src import visualize

visualize.plot_sentiment_distribution(
    tables["sentiment_distribution"],
    "Distribusi Sentimen Ulasan (berdasarkan teks)",
    "03_distribusi_sentimen_teks",
)

WindowsPath('D:/hikam/tugas DataMining/results/figures/03_distribusi_sentimen_teks.png')

In [8]:
visualize.plot_score_distribution(scored["skor_sentimen"], "03_distribusi_skor")

WindowsPath('D:/hikam/tugas DataMining/results/figures/03_distribusi_skor.png')

## 3.4 Perbandingan dengan rating

Inilah bagian yang diminta metodologi: menguji apakah sentimen hasil analisis
teks sesuai dengan rating yang diberikan pelanggan.

In [9]:
tables["cross_rating"]

sentimen_teks,Rating,Positif,Netral,Negatif
0,1 bintang,128,51,200
1,2 bintang,115,38,89
2,3 bintang,380,94,161
3,4 bintang,1110,278,184
4,5 bintang,2877,814,276


In [10]:
tables["cross_rating_pct"]

sentimen_teks,Rating,Positif,Netral,Negatif
0,1 bintang,33.77,13.46,52.77
1,2 bintang,47.52,15.70,36.78
2,3 bintang,59.84,14.80,25.35
3,4 bintang,70.61,17.68,11.70
4,5 bintang,72.52,20.52,6.96


In [11]:
visualize.plot_cross_rating(
    tables["cross_rating"],
    "03_sentimen_vs_rating",
    "Sentimen Teks vs Rating Pelanggan",
)

WindowsPath('D:/hikam/tugas DataMining/results/figures/03_sentimen_vs_rating.png')

## 3.5 Metrik kesesuaian

Rating diperlakukan sebagai acuan (1-2 → Negatif, 3 → Netral, 4-5 → Positif).

In [12]:
from src import evaluate

metrics = evaluate.classification_metrics(scored)
io_utils.save_csv(metrics, config.TABLES_DIR / "03_metrik_sentimen.csv")
metrics

,Metrik,Nilai
0,Accuracy,0.6431
1,Macro Precision,0.4187
2,Macro Recall,0.4444
3,Macro F1,0.4206
4,Weighted F1,0.6842
5,Cohen's Kappa,0.1446
6,Precision Positif,0.8649
7,Recall Positif,0.7198
8,F1 Positif,0.7857
9,Support Positif,5539.0000


In [13]:
per_class = evaluate.per_class_frame(scored)
io_utils.save_csv(per_class, config.TABLES_DIR / "03_metrik_per_kelas.csv")
per_class

,Kelas,Precision,Recall,F1-Score,Support
0,Positif,0.8649,0.7198,0.7857,5539
1,Netral,0.0737,0.1480,0.0984,635
2,Negatif,0.3176,0.4654,0.3775,621


In [14]:
cm = evaluate.confusion_frame(scored)
io_utils.save_csv(cm, config.TABLES_DIR / "03_confusion_matrix.csv")
visualize.plot_confusion(cm, "03_confusion_matrix")
cm

,,Prediksi (teks): Positif,Prediksi (teks): Netral,Prediksi (teks): Negatif
0,Aktual (rating): Positif,3987,1092,460
1,Aktual (rating): Netral,380,94,161
2,Aktual (rating): Negatif,243,89,289


## 3.6 Analisis gap: *hidden complaint* dan *hidden praise*

Bagian paling berguna untuk pengambilan keputusan: ulasan dengan rating tinggi
tetapi isinya keluhan, dan sebaliknya.

In [15]:
gap = tables["gap"]
io_utils.save_csv(gap, config.TABLES_DIR / "03_analisis_gap.csv")
gap

,Kategori,Jumlah,Persentase
0,Hidden complaint (rating 4-5 tapi teks negatif),460,6.77
1,Hidden praise (rating 1-2 tapi teks positif),243,3.58
2,Total ulasan dibandingkan,6795,100.00


### Contoh *hidden complaint* (rating 4-5, teks negatif)

Ini adalah keluhan yang **tidak akan pernah terlihat** jika hanya memakai rating.

In [16]:
hidden = scored[(scored["Rating"] >= 4) & (scored["sentimen_teks"] == "Negatif")]
hidden_cols = ["Rating", "Review", "skor_sentimen"]
hidden.sort_values("skor_sentimen")[hidden_cols].head(10)

,Rating,Review,skor_sentimen
3678,4,UPDATE : terakhir kesini mereka udah full operasional dari staff dan parkirnya dah dit...,-42.0
4090,4,Waow...itu yg terlintas di pikiran memasuki area cafe ini...dari depan dah kliatan ser...,-28.0
820,4,"So-so.<br>Pertama kali banget setelah beberapa kali ke Jogja, mampir ke Silol. Ke sini...",-23.5
1924,4,Udara rooftopnya lumayan dingin sebab angin terasa semilir sekali di sini. Saranku kal...,-23.5
1512,5,INFO UNTUK TEMAN2 YANG BELUM TAHU KALAU SILOL CAFE CLOSE ORDER JAM 12 MALAM SEMENJAK P...,-19.0
292,5,Lama ga kesini... Ee skrng harganya ngeri... Pesen carbonara piring gede tp isinya dik...,-14.0
2512,5,"Duduk di lantai 2 sambil nugas, ngopi latte, dan nyemilin jamur crispy. Good vibes bgt...",-13.5
6133,5,Cafe baru di Jogja yg terletak di jl cik ditiro. Berseberangan dengan toga mas kota ba...,-12.5
1599,4,"Tempat cukup bersih, pelayanan juga cukup baik. Sebelum masuk kita sudah disediakan wa...",-12.0
4638,5,Nice place to hang out with friends or being alone. The rooms haves a nostalgic feelin...,-12.0


### Contoh *hidden praise* (rating 1-2, teks positif)

In [17]:
praise = scored[(scored["Rating"] <= 2) & (scored["sentimen_teks"] == "Positif")]
praise.sort_values("skor_sentimen", ascending=False)[hidden_cols].head(10)

,Rating,Review,skor_sentimen
2419,1,"Pertama kali saya datang ke silol, saya mengekspektasikan sesuatu yang tinggi pada tem...",59.0
6092,1,"ada beberapa makanan yang tidak fresh, seperti waktu itu beli mendoan, mendoan yang di...",38.0
4138,2,"Servis nol, mulai dari kitchen dan server..pesen salad dtgnya belakangan dan server ng...",30.1
468,1,"Makanannya gosong, juicenya terlalu banyak air. Untuk makanan ga worth it sm sekali. U...",30.0
4161,1,"pelayanan nya masih jauh dari ramah, first impressionnya kurang baik, mohon diperbaiki...",27.0
1439,2,Tolong ditingkatkan lagi kebersihan makananya. Saya pesan caesar salad ada ulatnya. Ud...,21.5
233,1,Pelayannya<br>Ga ada yg Ramahh<br>Apalagi yg cewe<br>Ga enakk bgtt<br>Mukanyaa di liha...,21.0
6723,1,"Bgst, bjgn, asw. Pelayanan mantabb, kopii porsi besar, rasa nikmat. Nunggu in kopi cep...",21.0
3826,2,"konsep tempat ini bagus,interior dan dekor juga bagus,instagramable buat yang hobi pos...",20.6
6782,2,Mungkin karena masih baru di Kota baru jadi masih ada keterlambatan penyajian pesanann...,20.0


## 3.7 Contoh ulasan paling ekstrem

In [18]:
tables["extremes"][["Kategori", "Rating", "sentimen_teks", "skor_sentimen", "Review"]]

,Kategori,Rating,sentimen_teks,skor_sentimen,Review
0,Skor tertinggi,5,Positif,63.0,Silol Kopi & Eatery salah satu tempat nongkrong di Jogja. Suasana cozy dengan desain i...
1,Skor tertinggi,5,Positif,62.5,"Saat malam hari di Yogyakarta, saya berkunjung ke Silol Coffee & Eatery, salah satu te..."
2,Skor tertinggi,1,Positif,59.0,"Pertama kali saya datang ke silol, saya mengekspektasikan sesuatu yang tinggi pada tem..."
3,Skor tertinggi,5,Positif,58.0,Super Mantap !!!<br><br>Silol Kopi Top Banget...<br>Tempat yang recommended untuk hang...
4,Skor tertinggi,5,Positif,52.0,"Super rame,super keren,super luas dan super asyik.. tempat nongkrong dijogja yg buka 2..."
5,Skor tertinggi,5,Positif,46.6,"Tempat yang asyik buat ngopi di jogja<br>Suasananya enak, ada live music dan musiknya ..."
6,Skor tertinggi,5,Positif,46.0,Salah satu tempat ngopi & nongki yang wajib dikunjungi dan dicoba pas lagi jogja. Temp...
7,Skor tertinggi,5,Positif,46.0,Cafe keren di tengah kota yogyakarta... tempat nya menarik ada tiga lantai.. buka 24 j...
8,Skor terendah,1,Negatif,-56.6,JANGAN MAKAN DI SINI..<br><br>Saya menyesal ke Silol ini tanpa baca reviewnya dulu.. d...
9,Skor terendah,1,Negatif,-56.0,VERY BAD SERVICE!<br>1. Pesan Conpana yang dateng Macchiato<br>2. Pesen milkshake oreo...


## 3.8 Kesimpulan Tahap 3

- Sentimen dihitung dari **teks**, rating hanya pembanding.
- Tingkat kesesuaian teks vs rating dihitung pada tabel metrik di atas.
  Nilai yang tidak sempurna bukan kegagalan, melainkan temuan: rating bintang
  tidak selalu mencerminkan isi ulasan.
- Jumlah *hidden complaint* menjadi indikator bahwa analisis teks memberi
  informasi tambahan yang tidak tersedia dari rating.
- Hasil level ulasan ini belum cukup untuk menjawab "aspek mana yang perlu
  diperbaiki" karena satu ulasan bisa memuat pujian dan keluhan sekaligus.
  Tahap berikutnya memecah sentimen **per aspek layanan**.